<a href="https://colab.research.google.com/github/itzjiawei/SC4000_ML/blob/decision_tree/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!git clone https://github.com/itzjiawei/SC4000_ML.git

fatal: destination path 'SC4000_ML' already exists and is not an empty directory.


In [44]:
import pandas as pd
import os

file_path = 'SC4000_ML/data/processed/men_processed.csv'

if os.path.exists(file_path):
    df_men_processed = pd.read_csv(file_path)
    print("Successfully loaded 'men_processed.csv'!")
    display(df_men_processed.head())
else:
    print(f"File not found at: {file_path}. Please check the repository structure.")

Successfully loaded 'men_processed.csv'!


,Season,Team1,Team2,SeedDiff,AvgScoringMarginDiff,WinRateDiff,EFGDifferentialDiff,Target
0,2003,1411,1421,0,9.208046,0.151724,0.045943,0
1,2003,1112,1436,-15,10.309113,0.237685,0.044359,1
2,2003,1113,1272,3,-1.896552,-0.172414,-0.024137,1
3,2003,1141,1166,5,-8.805643,-0.085684,-0.020792,1
4,2003,1143,1301,-1,0.324138,0.124138,-0.007780,1


Accuracy

In [45]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Filter the dataset based on seasons
train_df = df_men_processed[(df_men_processed['Season'] >= 2003) & (df_men_processed['Season'] <= 2016)].copy()
test_df = df_men_processed[df_men_processed['Season'] == 2017].copy()

# 2. Define features and target
features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
target = 'Target'

X_train = train_df[features]
y_train = train_df[target]
X_test = test_df[features]
y_test = test_df[target]

# 3. Initialize and train the Decision Tree
clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

# 4. Predict on the 2017 test set
y_pred = clf.predict(X_test)

# 5. Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}\n")
print(f"Decision Tree Accuracy on 2017 Season: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Training set shape: (914, 4)
Testing set shape: (67, 4)

Decision Tree Accuracy on 2017 Season: 0.7015

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.65      0.69        34
           1       0.68      0.76      0.71        33

    accuracy                           0.70        67
   macro avg       0.70      0.70      0.70        67
weighted avg       0.70      0.70      0.70        67



In [46]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Include 2017 along with 2018, 2019, and 2021
test_years = [2017, 2018, 2019, 2021]
features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
target = 'Target'

combined_results = []

for year in test_years:
    # Training data is all historical seasons before the test year
    train_df = df_men_processed[df_men_processed['Season'] < year].copy()
    test_df = df_men_processed[df_men_processed['Season'] == year].copy()

    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

    # Train Decision Tree Classifier
    clf = DecisionTreeClassifier(random_state=42, max_depth=5)
    clf.fit(X_train, y_train)

    # Predict using Decision Tree
    dt_preds = clf.predict(X_test)
    dt_correct = np.sum(dt_preds == y_test)
    dt_accuracy = accuracy_score(y_test, dt_preds)

    # Predict using Dummy Model (Always predicts Team 1 Wins -> Target = 1)
    dummy_preds = np.ones(len(y_test))
    dummy_correct = np.sum(dummy_preds == y_test)
    dummy_accuracy = accuracy_score(y_test, dummy_preds)

    combined_results.append({
        'Season': year,
        'Games Tested': len(y_test),
        'DT Correct': dt_correct,
        'DT Accuracy': dt_accuracy,
        'Dummy Correct': dummy_correct,
        'Dummy Accuracy': dummy_accuracy
    })

# Display the combined results
combined_results_df = pd.DataFrame(combined_results)
print("Comparison of Decision Tree vs. Dummy Baseline Model (Including 2017):\n")
display(combined_results_df)

Comparison of Decision Tree vs. Dummy Baseline Model (Including 2017):



,Season,Games Tested,DT Correct,DT Accuracy,Dummy Correct,Dummy Accuracy
0,2017,67,47,0.701493,33,0.492537
1,2018,67,46,0.686567,31,0.462687
2,2019,67,46,0.686567,29,0.432836
3,2021,66,46,0.696970,32,0.484848


In [47]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

test_years = [2017, 2018, 2019, 2021]
features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
target = 'Target'

# We will test Decision Tree depths from 4 to 10
depths_to_test = range(4, 11)
depth_results = []

for depth in depths_to_test:
    row_result = {'Max Depth': depth}
    total_correct = 0
    total_games = 0

    for year in test_years:
        # Train on all seasons before the target year
        train_df = df_men_processed[df_men_processed['Season'] < year].copy()
        test_df = df_men_processed[df_men_processed['Season'] == year].copy()

        X_train = train_df[features]
        y_train = train_df[target]
        X_test = test_df[features]
        y_test = test_df[target]

        clf = DecisionTreeClassifier(random_state=42, max_depth=depth)
        clf.fit(X_train, y_train)

        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)

        row_result[f'{year} Accuracy'] = acc
        total_correct += np.sum(preds == y_test)
        total_games += len(y_test)

    row_result['Overall Accuracy'] = total_correct / total_games
    depth_results.append(row_result)

# Create DataFrame of the performance metrics
depth_comparison_df = pd.DataFrame(depth_results)
print("Decision Tree Performance Across Different Max Depths (4 to 10):\n")
display(depth_comparison_df)

Decision Tree Performance Across Different Max Depths (4 to 10):



,Max Depth,2017 Accuracy,2018 Accuracy,2019 Accuracy,2021 Accuracy,Overall Accuracy
0,4,0.731343,0.701493,0.716418,0.696970,0.711610
1,5,0.701493,0.686567,0.686567,0.696970,0.692884
2,6,0.716418,0.656716,0.641791,0.636364,0.662921
3,7,0.746269,0.611940,0.656716,0.606061,0.655431
4,8,0.731343,0.626866,0.716418,0.606061,0.670412
5,9,0.746269,0.567164,0.701493,0.636364,0.662921
6,10,0.746269,0.597015,0.746269,0.636364,0.681648


## **Evaluate using Brier Score for Men**


### S1: Original 4 features
- 2017: Brier Score: 0.1779 (Accuracy: 76%)
- 2018: Brier Score: 0.2121 (Accuracy: 69%)
- 2019: Brier Score: 0.1726 (Accuracy: 70%)
- 2021: Brier Score: 0.2205 (Accuracy: 68%)
### Average: 0.1958 (Average Accuracy: 70.75%)

In [48]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, brier_score_loss

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

# Search for the file in the workspace
file_name = 'men_processed.csv'
file_path = find_file(file_name, '.')

if file_path and os.path.exists(file_path):
    print(f"Found file at: {file_path}")
    df_men_processed = pd.read_csv(file_path)
else:
    # Fallback to cloning the repository again in case it was lost or removed
    print("File not found. Attempting to clone repository...")
    os.system('git clone https://github.com/itzjiawei/SC4000_ML.git')
    file_path = find_file(file_name, '.')
    if file_path:
        df_men_processed = pd.read_csv(file_path)
        print(f"Successfully found and loaded file after cloning: {file_path}")
    else:
        raise FileNotFoundError(f"Could not find {file_name} in the workspace.")

# Define target years and features based on the previous working examples
test_years = [2017, 2018, 2019, 2021]
features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
target = 'Target'

print("=== Random Forest Predictions & Brier Scores ===\n")

for year in test_years:
    # Split data chronologically
    train_df = df_men_processed[df_men_processed['Season'] < year].copy()
    test_df = df_men_processed[df_men_processed['Season'] == year].copy()

    if len(test_df) == 0:
        print(f"Skipping Season {year}: No test data available.\n")
        continue

    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

    # Train Random Forest Classifier
    rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
    rf_clf.fit(X_train, y_train)

    # Predict binary targets and probabilities
    y_pred = rf_clf.predict(X_test)
    # Probability of class 1 (Team 1 Winning)
    y_prob = rf_clf.predict_proba(X_test)[:, 1]

    # Calculate Brier Score
    brier_score = brier_score_loss(y_test, y_prob)

    print(f"==================================================")
    print(f"Season {year} Evaluation")
    print(f"==================================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"Brier Score: {brier_score:.4f}\n")

Found file at: ./SC4000_ML/data/processed/men_processed.csv
=== Random Forest Predictions & Brier Scores ===

Season 2017 Evaluation
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.65      0.73        34
           1       0.71      0.88      0.78        33

    accuracy                           0.76        67
   macro avg       0.78      0.76      0.76        67
weighted avg       0.78      0.76      0.76        67

Brier Score: 0.1779

Season 2018 Evaluation
Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.78      0.73        36
           1       0.69      0.58      0.63        31

    accuracy                           0.69        67
   macro avg       0.69      0.68      0.68        67
weighted avg       0.69      0.69      0.68        67

Brier Score: 0.2121

Season 2019 Evaluation
Classification Report:
              precision    recall  f1-score   support

 

### S2: 4 features + MasseyRankDiff
- 2017: Brier Score = 0.1899
- 2018: Brier Score = 0.2236
- 2019: Brier Score = 0.1630
- 2021: Brier Score = 0.2598
### Average: 0.2091

In [49]:
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, brier_score_loss

massey_file_path = 'SC4000_ML/data/processed/men_massey.csv'

if os.path.exists(massey_file_path):
    df_massey = pd.read_csv(massey_file_path)
    print("Successfully loaded 'men_massey.csv'!")

    # Dynamically find 'MasseyRankDiff' ignoring case
    massey_col = None
    for col in df_massey.columns:
        if col.lower() == 'masseyrankdiff':
            massey_col = col
            break

    if massey_col:
        print(f"Using original untransformed column: '{massey_col}'")

        features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff', massey_col]
        target = 'Target'
        test_years = [2017, 2018, 2019, 2021]

        for year in test_years:
            train_df = df_massey[df_massey['Season'] < year].copy()
            test_df = df_massey[df_massey['Season'] == year].copy()

            train_df = train_df.dropna(subset=features + [target])
            test_df = test_df.dropna(subset=features + [target])

            X_train = train_df[features]
            y_train = train_df[target]
            X_test = test_df[features]
            y_test = test_df[target]

            clf = DecisionTreeClassifier(random_state=42, max_depth=5)
            clf.fit(X_train, y_train)

            y_pred = clf.predict(X_test)
            y_prob = clf.predict_proba(X_test)[:, 1]
            brier_score = brier_score_loss(y_test, y_prob)

            print(f"==================================================")
            print(f"Evaluation for Season {year} (with original '{massey_col}')")
            print(f"==================================================")
            print(classification_report(y_test, y_pred))
            print(f"Brier Score: {brier_score:.4f}")
            print("\n")
    else:
        print("MasseyRankDiff column not found.")
else:
    print(f"Error: File not found at {massey_file_path}.")

Successfully loaded 'men_massey.csv'!
Using original untransformed column: 'MasseyRankDiff'
Evaluation for Season 2017 (with original 'MasseyRankDiff')
              precision    recall  f1-score   support

           0       0.89      0.47      0.62        34
           1       0.63      0.94      0.76        33

    accuracy                           0.70        67
   macro avg       0.76      0.70      0.69        67
weighted avg       0.76      0.70      0.68        67

Brier Score: 0.1899


Evaluation for Season 2018 (with original 'MasseyRankDiff')
              precision    recall  f1-score   support

           0       0.68      0.69      0.68        36
           1       0.63      0.61      0.62        31

    accuracy                           0.66        67
   macro avg       0.65      0.65      0.65        67
weighted avg       0.66      0.66      0.66        67

Brier Score: 0.2236


Evaluation for Season 2019 (with original 'MasseyRankDiff')
              precision    rec

### S3: 4 features + elo
- 2017: Brier Score = 0.2494 (Accuracy: 63%)
- 2018: Brier Score = 0.2381 (Accuracy: 69%)
- 2019: Brier Score = 0.1853 (Accuracy: 69%)
- 2021: Brier Score = 0.2241 (Accuracy: 70%)
### Average: 0.2242 (Average Accuracy: 67.75%)

In [50]:
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, brier_score_loss

processed_file_path = 'SC4000_ML/data/processed/men_processed.csv'
elo_file_path = 'SC4000_ML/data/processed/men_elo.csv'

if os.path.exists(processed_file_path) and os.path.exists(elo_file_path):
    # Load datasets
    df_processed = pd.read_csv(processed_file_path).copy()
    df_elo = pd.read_csv(elo_file_path).copy()

    print("Successfully loaded 'men_processed.csv' and 'men_elo.csv'!")

    # Merge for Team1 Elo
    df_merged = pd.merge(
        df_processed,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team1_Elo'}).drop(columns=['TeamID'])

    # Merge for Team2 Elo
    df_merged = pd.merge(
        df_merged,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team2_Elo'}).drop(columns=['TeamID'])

    # Calculate EloDiff = Team1 Elo - Team2 Elo
    df_merged['EloDiff'] = df_merged['Team1_Elo'] - df_merged['Team2_Elo']
    print("Successfully computed 'EloDiff' and merged dataframes!")

    # Use 4 base features + EloDiff
    features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff', 'EloDiff']
    target = 'Target'
    test_years = [2017, 2018, 2019, 2021]

    for year in test_years:
        train_df = df_merged[df_merged['Season'] < year].copy()
        test_df = df_merged[df_merged['Season'] == year].copy()

        train_df = train_df.dropna(subset=features + [target])
        test_df = test_df.dropna(subset=features + [target])

        X_train = train_df[features]
        y_train = train_df[target]
        X_test = test_df[features]
        y_test = test_df[target]

        clf = DecisionTreeClassifier(random_state=42, max_depth=5)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]
        brier_score = brier_score_loss(y_test, y_prob)

        print(f"==================================================")
        print(f"Evaluation for Season {year} (with 4 features + 'EloDiff')")
        print(f"==================================================")
        print(classification_report(y_test, y_pred))
        print(f"Brier Score: {brier_score:.4f}")
        print("\n")
else:
    print("Error: One or both files were not found.")

Successfully loaded 'men_processed.csv' and 'men_elo.csv'!
Successfully computed 'EloDiff' and merged dataframes!
Evaluation for Season 2017 (with 4 features + 'EloDiff')
              precision    recall  f1-score   support

           0       0.62      0.68      0.65        34
           1       0.63      0.58      0.60        33

    accuracy                           0.63        67
   macro avg       0.63      0.63      0.63        67
weighted avg       0.63      0.63      0.63        67

Brier Score: 0.2494


Evaluation for Season 2018 (with 4 features + 'EloDiff')
              precision    recall  f1-score   support

           0       0.68      0.78      0.73        36
           1       0.69      0.58      0.63        31

    accuracy                           0.69        67
   macro avg       0.69      0.68      0.68        67
weighted avg       0.69      0.69      0.68        67

Brier Score: 0.2381


Evaluation for Season 2019 (with 4 features + 'EloDiff')
              pre

### S4: 4 features + MasseyRankDiff + Elo
- 2017: Brier Score: 0.2023 (Accuracy: 70%)
- 2018: Brier Score: 0.2352 (Accuracy: 67%)
- 2019: Brier Score: 0.2085 (Accuracy: 70%)
- 2021: Brier Score: 0.2483 (Accuracy: 70%)
### Average: 0.2236 (Average Accuracy: 69.25%)

In [51]:
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, brier_score_loss

massey_file_path = 'SC4000_ML/data/processed/men_massey.csv'
elo_file_path = 'SC4000_ML/data/processed/men_elo.csv'

if os.path.exists(massey_file_path) and os.path.exists(elo_file_path):
    # Load datasets
    df_massey = pd.read_csv(massey_file_path).copy()
    df_elo = pd.read_csv(elo_file_path).copy()

    print("Successfully loaded 'men_massey.csv' and 'men_elo.csv'!")

    # Dynamically handle MasseyRankDiff column
    massey_col = 'MasseyRankDiff'

    # Merge for Team1 Elo
    df_merged = pd.merge(
        df_massey,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team1_Elo'}).drop(columns=['TeamID'])

    # Merge for Team2 Elo
    df_merged = pd.merge(
        df_merged,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team2_Elo'}).drop(columns=['TeamID'])

    # Calculate EloDiff = Team1 Elo - Team2 Elo
    df_merged['EloDiff'] = df_merged['Team1_Elo'] - df_merged['Team2_Elo']
    print("Successfully computed 'EloDiff' and merged dataframes!")

    # Use original 'MasseyRankDiff' (massey_col)
    features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff', massey_col, 'EloDiff']
    target = 'Target'
    test_years = [2017, 2018, 2019, 2021]

    for year in test_years:
        train_df = df_merged[df_merged['Season'] < year].copy()
        test_df = df_merged[df_merged['Season'] == year].copy()

        train_df = train_df.dropna(subset=features + [target])
        test_df = test_df.dropna(subset=features + [target])

        X_train = train_df[features]
        y_train = train_df[target]
        X_test = test_df[features]
        y_test = test_df[target]

        clf = DecisionTreeClassifier(random_state=42, max_depth=5)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]
        brier_score = brier_score_loss(y_test, y_prob)

        print(f"==================================================")
        print(f"Evaluation for Season {year} (with '{massey_col}' + 'EloDiff')")
        print(f"==================================================")
        print(classification_report(y_test, y_pred))
        print(f"Brier Score: {brier_score:.4f}")
        print("\n")
else:
    print("Error: One or both files were not found.")

Successfully loaded 'men_massey.csv' and 'men_elo.csv'!
Successfully computed 'EloDiff' and merged dataframes!
Evaluation for Season 2017 (with 'MasseyRankDiff' + 'EloDiff')
              precision    recall  f1-score   support

           0       0.85      0.50      0.63        34
           1       0.64      0.91      0.75        33

    accuracy                           0.70        67
   macro avg       0.74      0.70      0.69        67
weighted avg       0.75      0.70      0.69        67

Brier Score: 0.2023


Evaluation for Season 2018 (with 'MasseyRankDiff' + 'EloDiff')
              precision    recall  f1-score   support

           0       0.68      0.72      0.70        36
           1       0.66      0.61      0.63        31

    accuracy                           0.67        67
   macro avg       0.67      0.67      0.67        67
weighted avg       0.67      0.67      0.67        67

Brier Score: 0.2352


Evaluation for Season 2019 (with 'MasseyRankDiff' + 'EloDiff')
  

### **Evaluation & Feature Importance Analysis (S1 - S4)**

To determine which features add value, we analyze the performance of our chronological validation setups across four target seasons (2017, 2018, 2019, 2021).

| Configuration | Added Features | Average Brier Score (Lower is Better) | Average Classification Accuracy (Higher is Better) |
| :--- | :--- | :---: | :---: |
| **S1** | Original 4 base features (*SeedDiff, AvgScoringMarginDiff, WinRateDiff, EFGDifferentialDiff*) | **0.1958** | **70.75%** |
| **S2** | Base 4 + *MasseyRankDiff* (using a Decision Tree) | **0.2091** | **69.00%** |
| **S3** | Base 4 + *EloDiff* (using a Decision Tree) | **0.2242** | **67.75%** |
| **S4** | Base 4 + *MasseyRankDiff* + *EloDiff* (using a Decision Tree) | **0.2236** | **69.25%** |

---

### **Key Insights & Feature Utility**

1. **Baseline is Strongest (S1):**
   - **S1** (the Random Forest trained strictly on the 4 base features) achieves the lowest Brier Score (**0.1958**) and the highest average accuracy (**70.75%**). This suggests that the initial subset of statistics already captures the core variance of team strength.

2. **Massey Rankings Performance (S2):**
   - Adding **MasseyRankDiff** (S2) increases the average Brier Score to **0.2091** and decreases accuracy slightly to **69.00%** when evaluated using a standard Decision Tree (max_depth=5). While Massey consensus ratings are powerful, they often contain redundant information with `SeedDiff` and `WinRateDiff` or can overfit when used inside shallow trees.

3. **Elo Ratings Performance (S3 & S4):**
   - Adding **EloDiff** on its own (S3) degrades the performance further to a Brier score of **0.2242** and the lowest average accuracy of **67.75%**.
   - Adding both **MasseyRankDiff and EloDiff** (S4) performs better than Elo alone (Brier: **0.2236**, Accuracy: **69.25%**), but still fails to outperform the S1 baseline.

### **Conclusion: Which Features are Useful?**
- **SeedDiff, AvgScoringMarginDiff, WinRateDiff, and EFGDifferentialDiff** remain the most robust and robustly generalized feature subset.
- **MasseyRankDiff** and **EloDiff** are highly correlated with our base features (multi-collinearity). When added to shallow decision trees, they seem to introduce noise or sub-optimal decision boundaries, causing slight overfitting on historical tournament samples.

## **Evaluate using Brier Score for Women**


### S1: Original 4 features
- 2017: Brier Score: 0.1719 (Accuracy: 81%)
- 2018: Brier Score: 0.1747 (Accuracy: 73%)
- 2019: Brier Score: 0.1302 (Accuracy: 81%)
- 2021: Brier Score: 0.1407 (Accuracy: 78%)
### Average: 0.1544 (Average Accuracy: 78.25%)

In [52]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, brier_score_loss

def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

# Search for the file in the workspace
file_name = 'women_processed.csv'
file_path = find_file(file_name, '.')

if file_path and os.path.exists(file_path):
    print(f"Found file at: {file_path}")
    df_women_processed = pd.read_csv(file_path)
else:
    # Fallback to cloning the repository again in case it was lost or removed
    print("File not found. Attempting to clone repository...")
    os.system('git clone https://github.com/itzjiawei/SC4000_ML.git')
    file_path = find_file(file_name, '.')
    if file_path:
        df_women_processed = pd.read_csv(file_path)
        print(f"Successfully found and loaded file after cloning: {file_path}")
    else:
        raise FileNotFoundError(f"Could not find {file_name} in the workspace.")

# Define target years and features based on the previous working examples
test_years = [2017, 2018, 2019, 2021]
features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff']
target = 'Target'

print("=== Random Forest Predictions & Brier Scores ===\n")

for year in test_years:
    # Split data chronologically
    train_df = df_women_processed[df_women_processed['Season'] < year].copy()
    test_df = df_women_processed[df_women_processed['Season'] == year].copy()

    if len(test_df) == 0:
        print(f"Skipping Season {year}: No test data available.\n")
        continue

    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

    # Train Random Forest Classifier
    rf_clf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
    rf_clf.fit(X_train, y_train)

    # Predict binary targets and probabilities
    y_pred = rf_clf.predict(X_test)
    # Probability of class 1 (Team 1 Winning)
    y_prob = rf_clf.predict_proba(X_test)[:, 1]

    # Calculate Brier Score
    brier_score = brier_score_loss(y_test, y_prob)

    print(f"==================================================")
    print(f"Season {year} Evaluation")
    print(f"==================================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"Brier Score: {brier_score:.4f}\n")

Found file at: ./SC4000_ML/data/processed/women_processed.csv
=== Random Forest Predictions & Brier Scores ===

Season 2017 Evaluation
Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.71      0.81        35
           1       0.72      0.93      0.81        28

    accuracy                           0.81        63
   macro avg       0.82      0.82      0.81        63
weighted avg       0.84      0.81      0.81        63

Brier Score: 0.1719

Season 2018 Evaluation
Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.72      0.71        29
           1       0.76      0.74      0.75        34

    accuracy                           0.73        63
   macro avg       0.73      0.73      0.73        63
weighted avg       0.73      0.73      0.73        63

Brier Score: 0.1747

Season 2019 Evaluation
Classification Report:
              precision    recall  f1-score   support


### S2: 4 features + elo
- 2017: Brier Score = 0.2077 (Accuracy: 78%)
- 2018: Brier Score = 0.2142 (Accuracy: 68%)
- 2019: Brier Score = 0.1646 (Accuracy: 78%)
- 2021: Brier Score = 0.1308 (Accuracy: 81%)
### Average: 0.1793 (Average Accuracy: 76.19%)

In [53]:
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, brier_score_loss, accuracy_score

# Helper to find file path
def find_file(name, path):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    return None

processed_file_path = find_file('women_processed.csv', '.')
elo_file_path = find_file('women_elo.csv', '.')

if not processed_file_path or not elo_file_path:
    # Fallback to cloning the repository again
    print("Files not found. Attempting to clone repository...")
    os.system('git clone https://github.com/itzjiawei/SC4000_ML.git')
    processed_file_path = find_file('women_processed.csv', '.')
    elo_file_path = find_file('women_elo.csv', '.')

if processed_file_path and elo_file_path:
    df_processed = pd.read_csv(processed_file_path).copy()
    df_elo = pd.read_csv(elo_file_path).copy()
    print("Successfully loaded women_processed.csv and women_elo.csv!")

    # Merge for Team1 Elo
    df_merged = pd.merge(
        df_processed,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team1_Elo'}).drop(columns=['TeamID'])

    # Merge for Team2 Elo
    df_merged = pd.merge(
        df_merged,
        df_elo[['Season', 'TeamID', 'FinalElo']],
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='inner'
    ).rename(columns={'FinalElo': 'Team2_Elo'}).drop(columns=['TeamID'])

    # Calculate EloDiff = Team1 Elo - Team2 Elo
    df_merged['EloDiff'] = df_merged['Team1_Elo'] - df_merged['Team2_Elo']
    print("Successfully computed 'EloDiff' and merged dataframes!")

    # Define features and target
    features = ['SeedDiff', 'AvgScoringMarginDiff', 'WinRateDiff', 'EFGDifferentialDiff', 'EloDiff']
    target = 'Target'
    test_years = [2017, 2018, 2019, 2021]

    brier_scores = []
    accuracies = []

    for year in test_years:
        train_df = df_merged[df_merged['Season'] < year].copy()
        test_df = df_merged[df_merged['Season'] == year].copy()

        train_df = train_df.dropna(subset=features + [target])
        test_df = test_df.dropna(subset=features + [target])

        X_train = train_df[features]
        y_train = train_df[target]
        X_test = test_df[features]
        y_test = test_df[target]

        clf = DecisionTreeClassifier(random_state=42, max_depth=5)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        brier_score = brier_score_loss(y_test, y_prob)
        accuracy = accuracy_score(y_test, y_pred)

        brier_scores.append(brier_score)
        accuracies.append(accuracy)

        print(f"==================================================")
        print(f"Women Evaluation for Season {year} (4 features + 'EloDiff')")
        print(f"==================================================")
        print(classification_report(y_test, y_pred))
        print(f"Brier Score: {brier_score:.4f} | Accuracy: {accuracy*100:.1f}%")
        print("\n")

else:
    print("Error loading women's data files.")

Successfully loaded women_processed.csv and women_elo.csv!
Successfully computed 'EloDiff' and merged dataframes!
Women Evaluation for Season 2017 (4 features + 'EloDiff')
              precision    recall  f1-score   support

           0       0.82      0.77      0.79        35
           1       0.73      0.79      0.76        28

    accuracy                           0.78        63
   macro avg       0.78      0.78      0.78        63
weighted avg       0.78      0.78      0.78        63

Brier Score: 0.2077 | Accuracy: 77.8%


Women Evaluation for Season 2018 (4 features + 'EloDiff')
              precision    recall  f1-score   support

           0       0.64      0.72      0.68        29
           1       0.73      0.65      0.69        34

    accuracy                           0.68        63
   macro avg       0.68      0.69      0.68        63
weighted avg       0.69      0.68      0.68        63

Brier Score: 0.2142 | Accuracy: 68.3%


Women Evaluation for Season 2019 (4 

### **Evaluation & Cross-Gender Comparison (S1 vs. S2)**

To determine the usefulness of adding Elo ratings, we compare the performance of our chronological validation setups (S1 vs. S2) across both the Men's and Women's datasets.

### **Performance Table Comparison**

| Dataset | Configuration | Added Features | 2017 Brier | 2018 Brier | 2019 Brier | 2021 Brier | Average Brier (Lower is Better) | Average Accuracy (Higher is Better) |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Men's** | **S1** (Baseline) | Original 4 Base Features | 0.1779 | 0.2121 | 0.1726 | 0.2205 | **0.1958** | **70.75%** |
| **Men's** | **S3** (S1 + Elo) | 4 Base Features + `EloDiff` | 0.2494 | 0.2381 | 0.1853 | 0.2241 | **0.2242** | **67.75%** |
| **Women's** | **S1** (Baseline) | Original 4 Base Features | 0.1719 | 0.1747 | 0.1302 | 0.1407 | **0.1544** | **78.25%** |
| **Women's** | **S2** (S1 + Elo) | 4 Base Features + `EloDiff` | 0.2077 | 0.2142 | 0.1646 | 0.1308 | **0.1793** | **76.19%** |

---

### **Key Insights & Comparative Analysis**

1. **Baseline superiority across genders:**
   - In both the men's and women's divisions, adding `EloDiff` to the model **degraded the overall metrics**.
   - For the men, the average Brier score rose from **0.1958 to 0.2242** (an increase of **0.0284**), with accuracy dropping by **3.00%**.
   - For the women, the average Brier score similarly rose from **0.1544 to 0.1793** (an increase of **0.0249**), with accuracy declining by **2.06%**.

2. **The impact of smaller data volume in the Women's Division:**
   - Women's basketball historically has fewer division-I teams and less comprehensive inter-conference matchup history than the men's side, leading to a smaller overall tournament dataset across the target years.
   - Despite the smaller data footprint, the women's models achieve significantly better absolute Brier scores (~0.15–0.18) and accuracies (~76–78%) than the men's models. This reflects the historical structural reality of women's college basketball, where top seeds are highly dominant and outcomes are more predictable compared to the higher-parity (and more upset-prone) men's tournament.

3. **Why Elo degrades performance:**
   - **Collinearity & Overfitting:** Elo ratings are strongly correlated with the seed differences (`SeedDiff`) and historical win rates (`WinRateDiff`) that are already in our 4 base features. Adding Elo to a shallow Decision Tree (max_depth=5) dilutes the importance of these highly stable base features and forces sub-optimal splits on redundant information.
   - **Parity sensitivity:** Because Elo is highly sensitive to recent game-by-game swings, it captures short-term variance that does not generalize well to postseason single-elimination dynamics, where raw season-long power metrics (scoring margins and efficient field goal differentials) are more robust predictors.